<a href="https://colab.research.google.com/github/InResponse-Counterspell/AI-101-Assignment-5/blob/main/Jeremy_Mathews_Assignment5a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U crewai
!pip install -q -U duckduckgo-search
!pip install -q -U crewai[tools]
!pip install -q -U google-generativeai

### Explanation of Cell 1 (`NYTr4e9yudDX`)

#### Code Explanation:
- `!pip install -q -U crewai`: Quietly installs or upgrades the core `crewai` package to build multi-agent workflows.
- `!pip install -q -U duckduckgo-search`: Quietly installs the DuckDuckGo search library for web searches.
- `!pip install -q -U crewai[tools]`: Installs extra dependencies and standard tools developed for CrewAI agents.
- `!pip install -q -U google-generativeai`: Quietly installs the Google GenAI SDK to interface with Gemini models.

#### Output:
This cell executes silently (due to `-q` flag) and installs/updates all necessary Python packages for running the multi-agent framework.

In [2]:
from google.colab import userdata, drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


### Explanation of Cell 2 (`OVyEDgUp3nZI`)

#### Code Explanation:
- `from google.colab import userdata, drive`: Imports the Google Colab modules needed to access secrets or mount Google Drive.
- `drive.mount("/content/drive", force_remount=True)`: Mounts your personal Google Drive into the Colab environment so files can be accessed or saved directly.

#### Output:
- `Mounted at /content/drive`: Confirms that the Google Drive has been successfully attached to `/content/drive`.

In [3]:
import os
from crewai import Agent, Task, Crew, Process, LLM
import google.generativeai as genai

from crewai.tools import BaseTool

# make sure to update with your Gemini API key
gemini_api_key = "xxx"

# Configure the API key
genai.configure(api_key=gemini_api_key)
#drive.mount('/content/drive')
file_path = '/content/NYC.csv' # Updated with the correct path of your uploaded file

try:
    with open(file_path, 'r') as f:
        csv_string = f.read()
    print("File content read successfully!")
    print(csv_string[:500]) # Print the first 500 characters to show the content
except FileNotFoundError:
    print(f"File not found at {file_path}. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred: {e}")

/tmp/ipykernel_16692/326190464.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


File content read successfully!
id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982154846191406,40.767936706542969,-73.964630126953125,40.765602111816406,N,455
id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415344238281,40.738563537597656,-73.999481201171875,40.731151580810547,N,663
id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979


### Explanation of Cell 3 (`HJNoRBkI1TY6`)

#### Code Explanation:
- `import os`: Imports the operating system module to work with directory paths.
- `from crewai import Agent, Task, Crew, Process, LLM`: Imports the core elements needed to configure agents, tasks, processes, and model backends in CrewAI.
- `import google.generativeai as genai`: Imports the legacy Google Generative AI module to configure API access.
- `from crewai.tools import BaseTool`: Imports the base template class needed to construct custom tools for agents.
- `gemini_api_key = "..."`: Saves your personal API authorization key into a variable.
- `genai.configure(api_key=gemini_api_key)`: Applies the API key configuration globally for Google's services.
- `file_path = '/content/NYC.csv'`: Points to the source NYC taxi dataset in your environment.
- `with open(...) as f`: Opens the CSV file, reads its entire contents into `csv_string`, and prints the first 500 characters to confirm a successful read.

#### Output:
- Shows a confirmation message: `File content read successfully!`
- Prints the header and the first few rows of the taxi trip dataset (containing vendor IDs, timestamps, coordinates, and trip durations).

In [4]:
# Use the recommended gemini-3.8-flash model
gemini_llm = LLM(
    model='gemini/gemini-3.8-flash',
    api_key=gemini_api_key,
    temperature=0.0  # Lower temperature for more consistent results.
)

# If issues persist, consider simplifying the tasks.

### Explanation of Cell 4 (`JXss11Ku1lEN`)

#### Code Explanation:
- `gemini_llm = LLM(...)`: Configures the main model backend for CrewAI to use `gemini/gemini-3.8-flash` via the CrewAI LLM wrapper, passing the API key and setting `temperature=0.0` to ensure deterministic, structured, and factual results.

#### Output:
This cell sets up the large language model backend silently in the background.

In [5]:
print(csv_string[:500])
#

id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982154846191406,40.767936706542969,-73.964630126953125,40.765602111816406,N,455
id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415344238281,40.738563537597656,-73.999481201171875,40.731151580810547,N,663
id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979


### Explanation of Cell 5 (`euqvkdqZ15Sv`)

#### Code Explanation:
- `print(csv_string[:500])`: Safely prints the first 500 characters of the loaded `csv_string` string to inspect structural details and columns.

#### Output:
- Prints the column headers and first three records of the NYC taxi dataset.

In [8]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
import pandas as pd

# Define a custom tool to read the CSV string
class CsvStringReaderTool(BaseTool):
    name: str = "CSV String Reader"
    description: str = "Reads the location data from the provided CSV string."
    csv_data: str

    def _run(self, query: str = None) -> str:
        """Reads a compact preview of the CSV data to keep token usage low."""
        # Send only the first 20 rows of the CSV to the agent to avoid hitting token limits
        lines = self.csv_data.strip().split('\n')
        return '\n'.join(lines[:20])

# Instantiate the custom tool with your csv_string
csv_reader_tool = CsvStringReaderTool(csv_data=csv_string)

# Define a smaller, token-friendly sample for the prompt description
csv_sample = '\n'.join(csv_string.strip().split('\n')[:10])

# Define one of your agents
data_analyst = Agent(
    role='Data analyst',
    goal='use the location data in the csv to find any key trends and insights.',
    backstory='You are a senoir data analyst that has 20 years of experience finding data that is valuable to your clients',
    verbose=True,
    allow_delegation=False,
    llm=gemini_llm, # Use the configured Gemini LLM
    tools=[csv_reader_tool] # Add the custom tool to the agent
)

# Define the task for the agent using the compact sample
analyze_data_task = Task(
    description=f"""Analyze the location data shown in this sample:

    {csv_sample}

    Your analysis should include:
    - Summary statistics for numerical columns (passenger_count, trip_duration, etc.).
    - Identification of any notable trends or patterns in trip durations.
    - Any other interesting observations you can draw from the dataset.
""",
    expected_output='A summary summarizing the key findings from location data analysis',
    agent=data_analyst
)

### Explanation of Cell 6 (`fe0e64fa`)

#### Code Explanation:
- `class CsvStringReaderTool(BaseTool)`: Defines a custom tool for the agents that reads and returns a limited 20-line block of the CSV file. Limiting the row count ensures we stay within the Gemini API's rate limits.
- `csv_sample = ...`: Slices the first 10 rows of the CSV to be sent directly to the agent's task description.
- `data_analyst = Agent(...)`: Instantiates the **Data Analyst Agent**, defining its role, objective, backstory, LLM backend, and granting it access to `csv_reader_tool`.
- `analyze_data_task = Task(...)`: Creates a task for the data analyst to review the CSV sample and identify trip patterns, trends, and statistics.

#### Output:
Configures the analyst agent and its analysis task in the memory space.

In [9]:
import nest_asyncio
# Apply the nest_asyncio patch to allow nested event loops in Jupyter/Colab
nest_asyncio.apply()

# Define another agent
stock_predictor = Agent(
    role='Overhead cost predictor',
    goal='Predict future overhead cost based on historical location data and identified trends.',
    backstory='You have indepth experience predicting client overhead cost for large scale companies',
    verbose=True,
    allow_delegation=False,
    llm=gemini_llm # Use the configured Gemini LLM
)

# Define the task for the stock price prediction agent
predict_stock_price_task = Task(
    description=f"""Using the provided location data sample:

{csv_sample}
    - And the insights from the data analysis report:

{{analyze_data_task.tasks_output}}

Predict potential future fuel costs for the companies based on the historical data and identified trends. Consider the relationships between average travel time and other overhead metrics (Maintaince, labor, etc.) in your predictions.

Give a predicted overhead cost for each company for two years from now, and compare that with the actual overhead cost.

Present your findings in a table that shows current overhead cost, and future overhead cost for each company
""",
    expected_output='A report containing overhead cost predictions for the companies based on the provided data and analysis, along with the reasoning for the predictions.',
    agent=stock_predictor,
    context=[analyze_data_task] # Add the analysis task as context
)

# Create a crew with both agents and tasks
financial_analysis_and_prediction_crew = Crew(
    agents=[data_analyst, stock_predictor],
    tasks=[analyze_data_task, predict_stock_price_task],
    process=Process.hierarchical, # Use hierarchical process for multiple agents
    manager_llm=gemini_llm # Specify the manager LLM
)

# Kick off the crew's work asynchronously to prevent Event Loop conflicts in Colab
print("## Starting the location analysis and overhead cost Crew asynchronously")
result = await financial_analysis_and_prediction_crew.kickoff_async()

# Print the results
print("## Final Report:")
print(result)

## Starting the location analysis and overhead cost Crew asynchronously


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data analyst                                                                                            │
│                                                                                                                 │
│  Task: Analyze the provided NYC taxi trip location dataset and produce a detailed report of findings            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data analyst                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # NYC Taxi Trip Location & Operational Data Analysis Report                                                    │
│                                                                                                                 │
│  **Prepared by:** Senior Data Analyst                                                                           │
│  **Focus:** Spatial-temporal dynamics, trip duration drivers, operational patterns, and geospatial clustering.  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│                                                                                                                 │
│  This evaluation analyzes the sample of NYC Yellow Taxi trip records collected across the first half of 2016.   │
│  Despite the compact sample size, the data exhibits distinct structural patterns characteristic of New York     │
│  City urban mobility:                                                                                           │
│                                                                                                                 │
│  * **Short-Hop Urban Dominance:** Over **77%** of trips are localized, short-distance hops within Manhattan     │
│  (median straight-line distance of **~1.49 km / 0.92 miles** and median duration of **7.38 minutes**).          │
│  * **Bimodal Duration Distribution:** While median trip duration is brisk (**443 seconds / ~7.4 minutes**),     │
│  the distribution is heavily right-skewed by two long-haul outliers: a cross-Manhattan north-to-south midday    │
│  traverse (**35.4 minutes**) and an inter-borough trip into Queens (**25.8 minutes**).                          │
│  * **Occupancy Profile:** Single-passenger trips dominate (**77.8%**), though multi-passenger demand spikes     │
│  during evening/nighttime leisure hours (up to 6 passengers).                                                   │
│  * **Zero Telemetry Lag:** 100% of trips recorded a `store_and_fwd_flag` of `'N'`, confirming instantaneous     │
│  transmission via cellular telemetry without offline vehicle buffering.                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Summary Statistics for Numerical Columns                                                                 │
│                                                                                                                 │
│  The table below provides parametric and non-parametric summary statistics across all numerical dimensions ($N  │
│  = 9$ sample trips):                                                                                            │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Overhead cost predictor                                                                                 │
│                                                                                                                 │
│  Task: Generate a comprehensive overhead cost prediction report for the taxi companies (Vendor 1 and Vendor 2)  │
│  based on the 2016 data sample, predicting overhead costs 2 years into the future (2018) including fuel,        │
│  maintenance, labor, and other operational overheads.                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Overhead cost predictor                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Overhead Cost Prediction Report: NYC Taxi Fleets (Vendor 1 vs. Vendor 2)                       │
│  **Projection Horizon:** 2016 Baseline to 2018 Forecast (2-Year Outlook)                                        │
│  **Target Entities:** Vendor 1 (Creative Mobile Technologies / CMT) & Vendor 2 (VeriFone Inc. / VTS)            │
│  **Prepared By:** Senior Overhead Cost Predictor & Fleet Economics Specialist                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│                                                                                                                 │
│  This report delivers a rigorous overhead cost evaluation and two-year predictive forecast (2016 baseline to    │
│  2018 projected) for New York City medallion taxi operations serviced by technology and fleet management        │
│  vendors: **Vendor 1 (Creative Mobile Technologies / CMT)** and **Vendor 2 (VeriFone Inc. / VTS)**.             │
│                                                                                                                 │
│  ### Core Findings                                                                                              │
│  1. **Divergent Duty Cycles:**                                                                                  │
│     - **Vendor 1** operates primarily in late-night/off-peak windows (22:30–01:00), characterized by shorter    │
│  trip durations ($\bar{t} = 7.00\text{ min}$), higher average transit speeds ($16.0\text{–}18.5\text{ km/h}$),  │
│  and lower traffic congestion, resulting in lower thermal and mechanical stress per mile.                       │
│     - **Vendor 2** operates across daytime and peak congestion windows (07:50–19:40), exhibiting longer haul    │
│  distances ($\bar{d} = 3.92\text{ street km}$), substantially longer trip durations ($\bar{t} = 15.10\text{     │
│  min}$), lower speeds ($8.9\text{–}13.3\text{ km/h}$), and high idle ratios ($>35\%$), which exacerbate hourly  │
│  labor and fuel burn.                                                                                           │
│                                                                                                                 │
│  2. **Cost Escalation Summary (2016 to 2018):**                                                                 │
│     - **Vendor 1 (CMT):** Total operational overhead increases from **$11.19** to **$13.11** across the 3-trip  │
│  sample (**+17.16%**), driven primarily by driver compensation adjustments and fuel price increases. Overhead   │
│  cost per operating hour increases from **$32.00/hr** to **$37.49/hr**.                                         │
│     - **Vendor 2 (VTS):** Total operational overhead increases from **$50.04** to **$59.39** across the 6-trip  │
│  sample (**+18.68%**), driven by severe congestion drag, fuel inflation, and escalating commercial vehicle      │
│  insurance. Overhead cost per operating hour increases 

## Final Report:
# Comprehensive Overhead Cost Prediction Report: NYC Taxi Fleets (Vendor 1 vs. Vendor 2)

**Projection Horizon:** 2016 Baseline to 2018 Forecast (2-Year Outlook)  
**Target Entities:** 
* **Vendor 1:** Creative Mobile Technologies (CMT)  
* **Vendor 2:** VeriFone Inc. (VTS)  

---

## 1. Executive Summary

This report provides a detailed operational cost evaluation and two-year predictive forecast (2016 baseline to 2018 projected) for New York City medallion taxi operations serviced by technology and fleet management vendors **Vendor 1 (Creative Mobile Technologies / CMT)** and **Vendor 2 (VeriFone Inc. / VTS)**.

### Core Findings & Divergent Operational Profiles
1. **Operational Profiles:**
   * **Vendor 1 (CMT):** In this sample, operations are concentrated in late-night/off-peak windows (22:30–01:00), featuring shorter average trip durations ($\bar{t} = 7.00\text{ minutes}$), higher average operating speeds ($10.67\text{ mph / } 17.17\text{ km/h}$), and lower idle 

### Explanation of Cell 7 (`xTTjHvmFbkRx`)

#### Code Explanation:
- `import nest_asyncio; nest_asyncio.apply()`: Patches the active event loop to allow asynchronous operations to run within the Colab environment without blocking.
- `stock_predictor = Agent(...)`: Defines the **Overhead Cost Predictor Agent** responsible for projecting business and travel overhead metrics.
- `predict_stock_price_task = Task(...)`: Creates the predictive task, linking it to the analyst's output via context integration.
- `financial_analysis_and_prediction_crew = Crew(...)`: Organizes agents and tasks into a hierarchical organization managed by the Gemini LLM.
- `await ...kickoff_async()`: Starts the crew asynchronously to solve the tasks sequentially.

#### Output:
- Starts the crew workflow and begins executing agent tasks using the optimized sample boundaries to successfully avoid token quota rate limits.

## Summary of Agent Output

The agents completed their workflow successfully using a token-optimized layout:

1. **Data Analyst Agent:** Evaluated the sample NYC taxi dataset, calculating trip duration characteristics and discovering patterns. Because the analysis was limited to the sample, the agent successfully summarized pickup/dropoff behaviors, ride distributions, and travel constraints without exceeding prompt token restrictions.
2. **Overhead Cost Predictor Agent:** Utilized the structured output from the Data Analyst to project overheads (maintenance, fuel, labor, operations). It created a 2-year forecast comparison of current vs. future overhead costs formatted in a clean table, detailing how average trip times and passenger counts influence transportation cost scales.

## Prompts Log

### Internal System & Task Prompts Used by Agents

#### 1. Data Analyst Task Prompt:
```text
Analyze the location data shown in this sample:

{csv_sample}

Your analysis should include:
- Summary statistics for numerical columns (passenger_count, trip_duration, etc.).
- Identification of any notable trends or patterns in trip durations.
- Any other interesting observations you can draw from the dataset.
```

#### 2. Cost Predictor Task Prompt:
```text
Using the provided location data sample:

{csv_sample}
- And the insights from the data analysis report:

{analyze_data_task.tasks_output}

Predict potential future fuel costs for the companies based on the historical data and identified trends. Consider the relationships between average travel time and other overhead metrics (Maintaince, labor, etc.) in your predictions.

Give a predicted overhead cost for each company for two years from now, and compare that with the actual overhead cost.

Present your findings in a table that shows current overhead cost, and future overhead cost for each company
```

---

### User Session Prompts
1. *"Load the NYC.csv file into Colab and use a multi-agent system to analyze it."*
2. *"Fix 429 Rate Limit (Quota Exceeded) errors encountered during execution."*
3. *"Explain each line of code, the outputs, summarize agent results, and log internal/session prompts."*